# 🔬 Lab 09: Automated System Evaluation & LLM-as-a-Judge

---

## 🎯 Executive Overview & Production Challenge
The biggest hurdle in deploying enterprise RAG is transitioning from **subjective "eyeball" evaluations** (*"looks good to me"*) to **automated, quantifiable, regression-tested metrics**.

### 📐 The RAG Triad of Evaluation:
A production RAG system has two core components: the **Retriever** and the **Generator**. We evaluate both independently and jointly:
1. **Context Relevance (Retriever Metric):** Are the retrieved chunks focused and relevant to the query, or polluted with noise?
2. **Faithfulness / Groundedness (Generator Metric):** Is every factual claim in the generated answer directly supported by the retrieved context? (Hallucination detection!)
3. **Answer Relevance (End-to-End Metric):** Does the generated answer directly address the user's question without digression?

```
                     [ User Question ]
                       │            │
             (Answer   │            │ (Context
            Relevance) │            │ Relevance)
                       ▼            ▼
             [ Generated Answer ] <──> [ Retrieved Context ]
                       (Faithfulness /
                        Groundedness)
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Load embedding model for semantic similarity scoring
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("[✓] Evaluator embedding model initialized.")


---
## Step 1: Building a Synthetic Golden Evaluation Benchmark
A golden dataset contains user questions, reference ground-truth contexts, and reference ground-truth answers.


In [ ]:
golden_benchmark = [
    {
        "question": "What is the emergency trip speed threshold for the gas turbine?",
        "ground_truth_context": "Section 1.1: Gas turbine normal operating speed N2 is 14,200 to 14,950 RPM with Level 2 ESD trip at 15,650 RPM.",
        "ground_truth_answer": "The gas turbine emergency shutdown trip speed is 15,650 RPM."
    },
    {
        "question": "How quickly must the anti-surge control valve open when Delta S drops below 0.08?",
        "ground_truth_context": "Section 2.1: The anti-surge control valve modulates to open within 350 milliseconds when Delta S falls below 0.08.",
        "ground_truth_answer": "The anti-surge valve must open within 350 milliseconds."
    },
    {
        "question": "What is the mandatory engineering Omanization percentage?",
        "ground_truth_context": "Section 4.3: Engineering and field maintenance operations require verified local Omanization ratios exceeding 68%.",
        "ground_truth_answer": "The mandatory Omanization staffing threshold is 68%."
    }
]
print(f"[✓] Prepared {len(golden_benchmark)} Golden Benchmark evaluation test cases.")


---
## Step 2: Implementing Automated LLM-as-a-Judge Evaluation Functions
We implement deterministic evaluators that compute the RAG Triad scores on a scale of $0.0$ to $1.0$.


In [ ]:
def calculate_context_relevance(question: str, retrieved_context: str) -> float:
    """Score how relevant the retrieved context is to the question."""
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    c_emb = embedder.encode([retrieved_context], normalize_embeddings=True)[0]
    return float(np.clip(np.dot(q_emb, c_emb), 0.0, 1.0))

def calculate_faithfulness(generated_answer: str, retrieved_context: str) -> float:
    """Check if claims in the generated answer are grounded in the retrieved context."""
    # Split answer into claims / sentences
    claims = [s.strip() for s in generated_answer.split('.') if len(s.strip()) > 10]
    if not claims:
        return 1.0
    c_emb = embedder.encode([retrieved_context], normalize_embeddings=True)[0]
    supported = 0
    for claim in claims:
        claim_emb = embedder.encode([claim], normalize_embeddings=True)[0]
        if np.dot(claim_emb, c_emb) > 0.45:
            supported += 1
    return float(supported / len(claims))

def calculate_answer_relevance(question: str, generated_answer: str) -> float:
    """Score whether the answer addresses the question directly."""
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    a_emb = embedder.encode([generated_answer], normalize_embeddings=True)[0]
    return float(np.clip(np.dot(q_emb, a_emb), 0.0, 1.0))

print("[✓] RAG Triad Evaluation functions compiled.")


---
## Step 3: Benchmarking Baseline Naive RAG vs. Advanced Hybrid RAG
Let's simulate evaluating two candidate RAG pipelines against our golden benchmark:
1. **Pipeline A (Naive RAG):** Prone to noise, low context relevance, occasional hallucination.
2. **Pipeline B (Advanced Hybrid RAG):** Strict filtering, re-ranking, and high faithfulness.


In [ ]:
# Pipeline A (Naive RAG simulation)
pipeline_a_results = [
    {
        "retrieved_context": "General operations overview for Fahud station including lube oil schedules.", # Noisy!
        "generated_answer": "Turbine trip speed is approximately 16,000 RPM based on standard designs." # Hallucination!
    },
    {
        "retrieved_context": "Anti-surge valve general details.",
        "generated_answer": "The valve opens fast." # Vague
    },
    {
        "retrieved_context": "Section 4.3: Engineering and field maintenance operations require Omanization exceeding 68%.",
        "generated_answer": "The Omanization threshold is 68%."
    }
]

# Pipeline B (Advanced RAG simulation)
pipeline_b_results = [
    {
        "retrieved_context": "Section 1.1: Gas turbine normal operating speed N2 is 14,200 to 14,950 RPM with Level 2 ESD trip at 15,650 RPM.",
        "generated_answer": "The gas turbine emergency trip speed is 15,650 RPM."
    },
    {
        "retrieved_context": "Section 2.1: The anti-surge control valve modulates to open within 350 milliseconds when Delta S falls below 0.08.",
        "generated_answer": "The anti-surge valve opens within 350 milliseconds."
    },
    {
        "retrieved_context": "Section 4.3: Engineering and field maintenance operations require verified local Omanization ratios exceeding 68%.",
        "generated_answer": "Mandatory Omanization ratio is 68% for engineering personnel."
    }
]

def evaluate_pipeline(name, benchmark, results):
    c_rel_list, faith_list, a_rel_list = [], [], []
    for test_case, res in zip(benchmark, results):
        q = test_case["question"]
        ctx = res["retrieved_context"]
        ans = res["generated_answer"]
        
        c_rel_list.append(calculate_context_relevance(q, ctx))
        faith_list.append(calculate_faithfulness(ans, ctx))
        a_rel_list.append(calculate_answer_relevance(q, ans))
        
    return {
        "pipeline": name,
        "context_relevance": float(np.mean(c_rel_list)),
        "faithfulness": float(np.mean(faith_list)),
        "answer_relevance": float(np.mean(a_rel_list))
    }

metrics_a = evaluate_pipeline("Naive RAG (Baseline)", golden_benchmark, pipeline_a_results)
metrics_b = evaluate_pipeline("Advanced Production RAG", golden_benchmark, pipeline_b_results)

print(f"{'Metric':<22} | {'Naive Baseline':<16} | {'Advanced Production':<16}")
print("-" * 62)
for k in ["context_relevance", "faithfulness", "answer_relevance"]:
    print(f"{k:<22} | {metrics_a[k]:<16.3f} | {metrics_b[k]:<16.3f}")


---
## Step 4: Visualizing Comparative Radar Charts
We plot a multi-axis radar chart showing the dramatic quantitative leap between naive and advanced RAG architectures.


In [ ]:
categories = ['Context Relevance', 'Faithfulness (Groundedness)', 'Answer Relevance']
values_a = [metrics_a['context_relevance'], metrics_a['faithfulness'], metrics_a['answer_relevance']]
values_b = [metrics_b['context_relevance'], metrics_b['faithfulness'], metrics_b['answer_relevance']]

# Repeat first value to close radar loop
values_a += values_a[:1]
values_b += values_b[:1]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True), dpi=120)
ax.plot(angles, values_a, color='#ef4444', linewidth=2, label='Naive RAG Baseline')
ax.fill(angles, values_a, color='#ef4444', alpha=0.15)

ax.plot(angles, values_b, color='#10b981', linewidth=2, label='Advanced Production RAG')
ax.fill(angles, values_b, color='#10b981', alpha=0.25)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), categories, fontsize=9)
ax.set_ylim(0.0, 1.0)
plt.title('RAG Triad Quantitative Evaluation Comparison', fontsize=11, fontweight='bold', pad=15)
plt.legend(loc='lower right', bbox_to_anchor=(1.25, 0.0), fontsize=8)
plt.tight_layout()
plt.show()


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Never deploy an updated embedding or chunking strategy without running golden benchmark evaluation.**
2. **Aim for Faithfulness > 0.95:** In high-risk engineering and compliance domains, unfaithful generated claims pose direct operational hazards.
